In [1]:
import numpy as np
import torch as th
import os
import os.path as osp
import pickle
import warnings
warnings.filterwarnings("ignore")
from scripts_utils import Parser
import diffuser.utils as utils
from diffuser.utils.arrays import to_np
from diffuser.datasets import object_rearrangement
from diffuser.datasets import AGENT
from AGENT_env import AGENT_env
from diffuser.datasets import highway
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from highway_env import register_highway_envs

register_highway_envs()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
pybullet build time: Nov 28 2023 23:51:11


In [2]:
%%bash
env="Overcooked"

# asymmetric_advantages coordination_ring counter_circuit_o_1order=random3, cramped_room forced_coordination=random0
layout="counter_circuit_o_1order" # asymmetric_advantages diverse_counter_circuit_6x5
pop=${layout}_comedi

num_agents=2
algo="population"
agent0_policy_name="comedi_oracle"
agent1_policy_name="proxy"
exp="eval-${agent0_policy_name}-${agent1_policy_name}"

path=/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration/mapbt/scripts/overcooked_population
population_yaml_path=${path}/pop_data/${pop}/comedi_oracle_vs_proxy.yml

export POLICY_POOL=${path}

In [3]:
from mapbt.algorithms.population.policy_pool import PolicyPool as Policy


In [4]:
def eval_overcooked(
    basedir, diffusion, dataset, renderer, dummy_cond, all_cond_features, all_cond_text,
    condition_guidance_w, device
):
    """
    Evaluate the overcooked model.
    """

In [5]:
if __name__ == "__main__":
    # args = Parser().parse_args('plan')
    device = th.device('cpu' if not th.cuda.is_available() else 'cuda')

    # load bc proxy
    population_yaml_path = "/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration/mapbt/scripts/overcooked_population/pop_data/counter_circuit_o_1order_comedi/comedi_oracle_vs_proxy.yml"
    policy = Policy(None, None, None, None, device=device)
    featurize_type = policy.load_population(population_yaml_path, evaluation=True)
    # policy.policy_pool['proxy'] is EvalPolicy object
    proxy_policy = policy.policy_pool['proxy']
    print("featurize_type: ", featurize_type)

featurize_type:  {'comedi_oracle': 'ppo', 'proxy': 'bc'}


In [6]:
def get_agent(population_yaml_path, policy_name):
    policy = Policy(None, None, None, None, device=device)
    featurize_type = policy.load_population(population_yaml_path, evaluation=True)
    # policy.policy_pool['proxy'] is EvalPolicy object
    policy = policy.policy_pool[policy_name]
    print("featurize_type: ", featurize_type)
    feat_type = featurize_type.get(policy_name, 'ppo')
    return policy, feat_type
gamma_path = "/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration"
yaml_path = os.path.join(gamma_path, "mapbt/scripts/overcooked_population/scripts/example_mep_counter_circuit/pop_for_vae_config.yml")
mep_policy, feature_type = get_agent(yaml_path, "sp10_init")
mep_policy, feature_type

featurize_type:  {'mep1_final': 'ppo', 'mep1_init': 'ppo', 'mep1_mid': 'ppo', 'mep2_final': 'ppo', 'mep2_init': 'ppo', 'mep2_mid': 'ppo', 'mep3_final': 'ppo', 'mep3_init': 'ppo', 'mep3_mid': 'ppo', 'mep4_final': 'ppo', 'mep4_init': 'ppo', 'mep4_mid': 'ppo', 'mep5_final': 'ppo', 'mep5_init': 'ppo', 'mep5_mid': 'ppo', 'mep6_final': 'ppo', 'mep6_init': 'ppo', 'mep6_mid': 'ppo', 'mep7_final': 'ppo', 'mep7_init': 'ppo', 'mep7_mid': 'ppo', 'mep8_final': 'ppo', 'mep8_init': 'ppo', 'mep8_mid': 'ppo', 'sp9_final': 'ppo', 'sp9_init': 'ppo', 'sp9_mid': 'ppo', 'sp10_final': 'ppo', 'sp10_init': 'ppo', 'sp10_mid': 'ppo'}


(<mapbt.algorithms.population.utils.EvalPolicy at 0x7fcaeb077850>, 'ppo')

In [7]:
def parse_args(args, parser):
    parser.add_argument("--old_dynamics", default=False, action='store_true', help="old_dynamics in mdp")
    parser.add_argument("--layout_name", type=str, default='cramped_room', help="Name of Submap, 40+ in choice. See /src/data/layouts/.")
    parser.add_argument('--num_agents', type=int,
                        default=1, help="number of players")
    parser.add_argument("--initial_reward_shaping_factor", type=float, default=1.0, help="Shaping factor of potential dense reward.")
    parser.add_argument("--reward_shaping_factor", type=float, default=1.0, help="Shaping factor of potential dense reward.")
    parser.add_argument("--reward_shaping_horizon", type=int, default=2.5e6, help="Shaping factor of potential dense reward.")
    parser.add_argument("--use_phi", default=False, action='store_true', help="While existing other agent like planning or human model, use an index to fix the main RL-policy agent.")  
    parser.add_argument("--use_hsp", default=False, action='store_true')   
    parser.add_argument("--random_index", default=False, action='store_true')
    parser.add_argument("--use_agent_policy_id", default=False, action='store_true', help="Add policy id into share obs, default False")
    parser.add_argument("--overcooked_version", default="old", type=str, choices=["new", "old"])
    parser.add_argument("--use_detailed_rew_shaping", default=False, action='store_true')
    parser.add_argument("--random_start_prob", default=0., type=float)
    parser.add_argument("--store_traj", default=False, action='store_true')
    # population
    parser.add_argument("--population_yaml_path", type=str, help="Path to yaml file that stores the population info.")
    
    # overcooked evaluation
    parser.add_argument("--agent0_policy_name", type=str, help="policy name of agent 0")
    parser.add_argument("--agent1_policy_name", type=str, help="policy name of agent 1")

    all_args = parser.parse_known_args(args)[0]

    return all_args

In [8]:
from mapbt.config import get_config
from mapbt.envs.overcooked.Overcooked_Env import Overcooked
import sys
parser = get_config()
args = sys.argv[1:]
all_args = parse_args(args, parser)

# assert all_args.algorithm_name == "population"
run_dir = '/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration/mapbt/scripts/results/Overcooked/counter_circuit_o_1order/population/eval-comedi_oracle-proxy/run14'

/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/site-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/site-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)


In [9]:
all_args.layout_name = 'counter_circuit_o_1order'
all_args.num_agents = 2
from argparse import Namespace
all_args = Namespace(activation_id=1, agent0_policy_name='comedi_oracle', agent1_policy_name='proxy', algorithm_name='population', attn_N=1, attn_heads=4, attn_size=64, aux_epoch=5, clip_param=0.2, clone_coef=1.0, cnn_layers_params=None, critic_lr=0.0005, critic_warmup_horizon=0, cuda=True, cuda_deterministic=True, data_chunk_length=10, dropout=0.0, entropy_coef=0.01, env_name='Overcooked', episode_length=400, eval_episodes=3, eval_interval=25, eval_stochastic=True, experiment_name='eval-comedi_oracle-proxy', explored_ratio_threshold=0.9, gae_lambda=0.95, gain=0.01, gamma=0.99, hidden_size=64, huber_delta=10.0, ifi=0.1, influence_layer_N=1, initial_reward_shaping_factor=1.0, layer_N=1, layout_name='counter_circuit_o_1order', log_interval=5, lr=0.0005, max_grad_norm=10.0, mlp_hidden_size=64, model_dir=None, n_eval_rollout_threads=3, n_render_rollout_threads=1, n_rollout_threads=32, n_training_threads=1, num_agents=2, num_env_steps=10000000.0, num_mini_batch=1, num_v_out=1, old_dynamics=True, opti_eps=1e-05, overcooked_version='old', policy_value_loss_coef=1, population_yaml_path='./overcooked_population/pop_data/counter_circuit_o_1order_comedi/comedi_oracle_vs_proxy.yml', ppo_epoch=15, random_index=False, random_start_prob=0.0, recurrent_N=1, render_episodes=5, reward_shaping_factor=1.0, reward_shaping_horizon=2500000.0, save_gifs=False, save_interval=1, seed=1, share_policy=True, stacked_frames=1, store_traj=False, tau=0.995, use_agent_policy_id=False, use_attn=False, use_attn_internal=True, use_average_pool=True, use_cat_self=True, use_centralized_V=True, use_clipped_value_loss=True, use_conv1d=False, use_detailed_rew_shaping=False, use_eval=False, use_feature_normalization=True, use_gae=True, use_hsp=False, use_huber_loss=True, use_influence_policy=False, use_linear_lr_decay=False, use_max_grad_norm=True, use_maxpool2d=False, use_naive_recurrent_policy=False, use_obs_instead_of_state=False, use_orthogonal=True, use_phi=False, use_policy_active_masks=True, use_policy_vhead=False, use_popart=False, use_proper_time_limits=False, use_recurrent_policy=True, use_render=False, use_single_network=False, use_stacked_frames=False, use_value_active_masks=True, use_valuenorm=True, use_wandb=False, user_name='user_name', value_loss_coef=1, wandb_name='wandb_name', wandb_tags=[], weight_decay=0)

In [10]:
from mapbt.envs.overcooked.Overcooked_Env import Overcooked
from mapbt.envs.env_wrappers import ChooseSubprocVecEnv

def make_eval_env(all_args, run_dir, nenvs=3):
    def get_env_fn(rank):
        def init_env():
            if all_args.env_name == "Overcooked":
                env = Overcooked(all_args, run_dir, rank=rank)
            else:
                print("Can not support the " +
                      all_args.env_name + "environment.")
                raise NotImplementedError
            env.seed(all_args.seed * 50000 + rank * 10000)
            return env
        return init_env
    return ChooseSubprocVecEnv([get_env_fn(i) for i in range(nenvs)])

envs = make_eval_env(all_args, run_dir)
agent_name = "sp10_init"
mep_policy, feature_type = get_agent(yaml_path, agent_name)
featurize_type = [['ppo', feature_type], ['ppo', feature_type], ['ppo', feature_type]]
envs.reset_featurize_type(featurize_type)

Using OvercookedEnv with the following parameters:
Using OvercookedEnv with the following parameters:Using OvercookedEnv with the following parameters:Namespace(activation_id=1, agent0_policy_name='comedi_oracle', agent1_policy_name='proxy', algorithm_name='population', attn_N=1, attn_heads=4, attn_size=64, aux_epoch=5, clip_param=0.2, clone_coef=1.0, cnn_layers_params=None, critic_lr=0.0005, critic_warmup_horizon=0, cuda=True, cuda_deterministic=True, data_chunk_length=10, dropout=0.0, entropy_coef=0.01, env_name='Overcooked', episode_length=400, eval_episodes=3, eval_interval=25, eval_stochastic=True, experiment_name='eval-comedi_oracle-proxy', explored_ratio_threshold=0.9, gae_lambda=0.95, gain=0.01, gamma=0.99, hidden_size=64, huber_delta=10.0, ifi=0.1, influence_layer_N=1, initial_reward_shaping_factor=1.0, layer_N=1, layout_name='counter_circuit_o_1order', log_interval=5, lr=0.0005, max_grad_norm=10.0, mlp_hidden_size=64, model_dir=None, n_eval_rollout_threads=3, n_render_rollout

Process Process-3:
Process Process-2:
Process Process-1:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/multip

In [11]:
obs, shared_obs, obs_avail = envs.reset([ True,  True,  True])

    
obs[0]

(array([[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0,

In [12]:
mep_policy.reset(num_envs=all_args.n_eval_rollout_threads, num_agents=2)
for e in range(all_args.n_eval_rollout_threads):
    mep_policy.register_control_agent(e=e, a=1)

In [16]:
obs_lst = [obs[e][a] for (e, a) in mep_policy.control_agents]

In [17]:
obs_lst

[array([[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0,

In [18]:
agents = mep_policy.control_agents
mep_policy.to('cuda:0')
proxy_action = mep_policy.step(np.stack(obs_lst, axis=0), agents)
print(proxy_action)

[[2]
 [1]
 [2]]


In [19]:
diffusion_loadpath='/mmfs1/gscratch/cse/jiayiy9/ftl-igm-overcooked/code/logs/overcooked/diffusion/defaults_H32_T100/20250415-114236'
diffusion_experiment = utils.load_diffusion(
    diffusion_loadpath,
    epoch='latest', seed=None,
)

[ utils/serialization ] Loaded config from /mmfs1/gscratch/cse/jiayiy9/ftl-igm-overcooked/code/logs/overcooked/diffusion/defaults_H32_T100/20250415-114236/dataset_config.pkl

[ utils/config ] Config: <class 'diffuser.datasets.overcooked.OvercookedSequenceDataset'>
    args: {'action_weight': 10,
 'attention': False,
 'batch_size': 32,
 'bucket': None,
 'chunk_length': 64,
 'clip_denoised': True,
 'condition_guidance_w': 1.0,
 'config': 'config.overcooked',
 'dataset': 'overcooked',
 'dataset_path': 'data/overcooked_dataset/counter_circuit_o_1order_mep/dataset.hdf5',
 'device': 'cuda',
 'diffusion': 'models.GaussianDiffusion',
 'dim_mults': (1, 2, 4, 8),
 'domain': 'overcooked',
 'ema_decay': 0.995,
 'episode_length': 400,
 'exp_name': 'diffusion/defaults_H32_T100',
 'gradient_accumulate_every': 2,
 'horizon': 32,
 'learning_rate': 0.0002,
 'loader': 'datasets.OvercookedSequenceDataset',
 'logbase': 'logs',
 'loss_discount': 1,
 'loss_type': 'l2',
 'loss_weights': None,
 'max_path_lengt

Using cache found in /mmfs1/gscratch/cse/jiayiy9/.cache/torch/hub/pytorch_vision_v0.10.0



[ utils/serialization ] Loading model epoch: 0



In [20]:
diffusion = diffusion_experiment.diffusion
diffusion.model.eval()
dataset = diffusion_experiment.dataset
renderer = diffusion_experiment.renderer   

In [21]:
# results path
basedir = diffusion_loadpath
dataset.policy_id

array([[0, 0],
       [0, 0],
       [0, 0],
       ...,
       [5, 5],
       [5, 5],
       [5, 5]])

In [22]:
def closed_loop_highway(eval_dir, diffusion, dataset, renderer, scenarios, device, n_concepts, mode='train', cond=None, n_samples_plot=8):
    if not osp.isdir(eval_dir): os.makedirs(eval_dir)
    n_demos_eval = 1 #per scenario
    for scenario_text,version in scenarios:
        SLOWER = 4 if scenario_text != 'intersection' else 0          
        env_name = f"{scenario_text.replace('_','-')}-v{version}"
        env_save_dir = f"{eval_dir}/{scenario_text}_{version}"
        video_folder = f"{env_save_dir}/videos"
        demos_pkl = f"{env_save_dir}/eval_{mode}.pkl"
        if mode=='train': 
            cond = dataset.generate_representation(scenario_text)
            cond=th.tensor(cond.reshape(1,-1)).to(device)
        else:
            cond = th.tensor(cond).to(device)
        # Make env
        env = gym.make(env_name, render_mode="rgb_array")
        env = RecordVideo(env, video_folder=video_folder, episode_trigger=lambda e: True) #default w/o episode_trigger: stop recording on terminated or truncated, causes issue with recorder.
        env.unwrapped.set_record_video_wrapper(env)
        # Run episodes
        trajs_obs, trajs_im, trajs_rew, trajs_done, trajs_trunc, trajs_info = [], [], [], [], [], []
        traj_num = 0
        while traj_num < n_demos_eval:
            (obs, info), done, truncated = env.reset(), False, False
            traj_obs, traj_im, traj_rew, traj_done, traj_trunc, traj_info = [obs], [], [], [], [], []
            all_inits = [obs]
            t = 0 # timestep
            while not (done or truncated):
                ######################
                # diffusion next state estimation
                init_s = th.tensor(dataset.normalize_init(traj_obs[-1]).flatten().reshape(1,-1)).to(device) #current obs
                with th.no_grad():
                    samples = diffusion.p_sample_loop(
                        shape=(1, dataset.horizon, dataset.observation_dim),
                        cond=cond,
                        dummy_cond=th.tensor(dataset.dummy_cond.reshape(1,-1)).to(device),
                        cond_obs=init_s,
                        compose=True if mode!='train' and n_concepts > 1 else False,
                        )
                s_t_1_unnorm = dataset.unnormalize(to_np(samples.trajectories)[0][min(t+1,dataset.horizon-1)]).squeeze() #future step
                # plot guidance
                guidance_dir = osp.join(env_save_dir, f'guidance_{traj_num}')
                if not osp.isdir(guidance_dir): os.mkdir(guidance_dir)
                highway.plot_traj(dataset.unnormalize(to_np(samples.trajectories)).squeeze(), traj_obs[-1], osp.join(guidance_dir, f'inv_model_diffusion_{t}.png'), dataset.n_vehicles, dataset.feat_dim, s_t_1_unnorm, cond_text=scenario_text) #diffusion guidance
                # inverse planning
                env_obs = env.observation_type.observe()
                inv_planning_obs = [] #n acts x n vehicles x features
                inv_planning_crashed = []
                for act in list(range(env.action_space.n)):
                    env_tmp = highway.safe_deepcopy_env(env.unwrapped)
                    env_tmp_obs = env_tmp.observation_type.observe()
                    assert (np.array(env_obs) == np.array(env_tmp_obs)).all()
                    obs, reward, done, truncated, info = env_tmp.step(act)
                    inv_planning_obs.append(obs)
                    inv_planning_crashed.append(info['crashed'])
                    assert (np.array(env_obs) == np.array(env.observation_type.observe())).all()
                    del env_tmp
                inv_planning_crashed = np.array(inv_planning_crashed)
                sim_vs_pred_states = np.linalg.norm(np.array(inv_planning_obs)[:,0,:]-s_t_1_unnorm, axis=1)
                if inv_planning_crashed.all():
                    best_act = SLOWER
                else:
                    best_act = np.random.choice(np.flatnonzero(sim_vs_pred_states == sim_vs_pred_states[~inv_planning_crashed].min())) #tie breaker best act not crashed (select act that gets us closest to diffusion pred w/o crashing)
                ######################
                obs, reward, done, truncated, info = env.step(best_act)
                traj_obs.append(obs)
                traj_rew.append(reward)
                traj_done.append(done)
                traj_trunc.append(truncated)
                traj_info.append(info) #speed, crashed, act
                img = env.render()
                traj_im.append(img)
                t += 1
            trajs_obs.append(traj_obs) # save as list, not numpy, not all same horizon (done or truncated)
            trajs_rew.append(traj_rew)
            trajs_done.append(traj_done)
            trajs_trunc.append(traj_trunc)
            trajs_info.append(traj_info) #has action
            trajs_im.append(traj_im)            
            traj_num += 1 
        # save, render, eval
        with open(demos_pkl, 'wb') as f: pickle.dump([trajs_obs, trajs_im, trajs_rew, trajs_done, trajs_trunc, trajs_info, scenario_text.replace('_',' ')], f)
        renderer.composite(osp.join(env_save_dir, f'closed_loop.png'), [np.array(samp) for samp in trajs_obs][:n_samples_plot], np.repeat(scenario_text,n_samples_plot), np.array(all_inits)[:n_samples_plot])
        highway.get_acc(scenario_text, trajs_rew, trajs_info, trajs_done, trajs_obs)
        env.close()

In [23]:
episode_length = 400
# def open_loop_overcooked(envs, cooperator_policy, basedir, diffusion, dataset, renderer, cond, condition_guidance_w, device, n_demos_eval=10, mode='train'):

init_obs, _, _ = envs.reset([ True,  True,  True])
all_obs, all_cond_text, all_inits, all_gt = [init_obs], [], [], []
agent_name = "sp10_init"
mep_policy.reset(num_envs=all_args.n_eval_rollout_threads, num_agents=2)
for e in range(all_args.n_eval_rollout_threads):
    mep_policy.register_control_agent(e=e, a=1)
mep_policy.to(device)

In [24]:
sample = dataset.__getitem__(0)
print(sample.trajectories[:, -1])

[0. 0. 2. 2. 0. 5. 1. 5. 0. 5. 1. 2. 5. 1. 2. 0. 2. 0. 2. 1. 3. 2. 2. 1.
 0. 2. 1. 1. 5. 3. 3. 1.]


In [25]:
cooperator_policy = mep_policy
mode = 'train'
n_demos_eval = 1
n_concepts = 1
obs = init_obs
nenvs = len(init_obs)
if mode=='train': 
    # cond = dataset.generate_representation(agent_name)
    cond = np.int64(5)
    cond = np.stack([cond] * nenvs, axis=0)
    print(cond)
    cond = th.tensor(cond).to(device)
    print(cond.shape)
else:
    cond = th.tensor(cond).to(device)

for _ in range(n_demos_eval):
    for t in range(1, episode_length):
        obs = all_obs[-1]

        eval_actions = np.full((nenvs, 2, 1), fill_value=0)

        # get cooperator action
        cooperator_obs_lst = [obs[e][a] for (e, a) in cooperator_policy.control_agents]
        cooperator_action = cooperator_policy.step(np.stack(cooperator_obs_lst, axis=0), agents, deterministic=True)
        print(cooperator_action, cooperator_action.shape)
        eval_actions[:, 1] = cooperator_action


        # get ego action
        ego_obs_lst = [dataset.normalize_init(obs[e][0]) for (e, a) in cooperator_policy.control_agents]
        
        sample = dataset.__getitem__(0)
        context_len, H, W, C = sample.conditions_obs.shape
        condition_obs = []

        for ego_obs in ego_obs_lst:
            cond_inputs = np.zeros((context_len, H, W, C), dtype=np.float32)
            cond_inputs[:-1] = cond_inputs[1:] # move backward
            cond_inputs[-1] = ego_obs
            condition_obs.append(cond_inputs)
        condition_obs = np.stack(condition_obs, axis=0)
        print(condition_obs.shape)
        condition_obs = th.tensor(condition_obs).to(device)

        dummy_cond = th.tensor(np.stack([sample.dummy_cond] * nenvs, axis=0)).to(device)
    
       # ego_obs = th.tensor().reshape(1,-1)).to(device) #current obs

        with th.no_grad():
            # cond: 1 digit id for cooperator
            # dummy_cond: 0
            # cond_obs: past observations, padded with zero
            samples = diffusion.p_sample_loop(
                shape=(nenvs, dataset.horizon, dataset.observation_dim + dataset.action_dim),
                cond=cond,
                dummy_cond=dummy_cond,
                cond_obs=condition_obs,
                compose=True if mode!='train' and n_concepts > 1 else False
            )

            # unnormalize the action. the code below is a placeholder.
            maxs = 5
            mins = 0

            ego_action = (samples[0][:, 0, -1]).cpu().numpy().reshape(-1,1) # [-1, 1] 
            ego_action = ego_action + 1
            ego_action /= 2
            ego_action = ego_action * (maxs - mins + 1e-5) + mins #[min,max]
            ego_action = ego_action.astype(int)
            print(ego_action, ego_action.shape)
            eval_actions[:, 0] = ego_action
        
        obs, done = envs.step(eval_actions)
        
        
        final_traj.append(obs.astype(np.float32))
        AGENT.plot_traj(np.array(final_traj), final_traj[0], osp.join(plot_dir,f'inv_model_obs.png'), cond_text=cond_text) #updating obs
        AGENT.plot_traj(dataset.unnormalize(to_np(samples.trajectories)).squeeze(), final_traj[0], osp.join(plot_dir,f'inv_model_diffusion_{t}.png'), s_t_1_unnorm, cond_text=cond_text) #diffusion guidance
        if done: break
    
    all_inits.append(dataset.unnormalize(sample.conditions_obs))
    all_samples.append(dataset.unnormalize(to_np(samples.trajectories)).squeeze())
    all_cond_text.append('') #dummy to plot
    all_gt.append(dataset.unnormalize(to_np(sample.trajectories)).squeeze())
# save, render
eval_dir = osp.join(basedir, f'eval_train_w_{condition_guidance_w}')
    # if not osp.isdir(eval_dir): os.makedirs(eval_dir)  
    # with open(osp.join(eval_dir, f'samples.pkl'), 'wb') as f: pickle.dump([all_samples, all_cond_text, all_inits, all_init_ims, all_gt], f)
    # renderer.composite(osp.join(eval_dir, f'render_samples.png'), np.array(all_samples), np.array(all_cond_text), np.array(all_inits), np.array(all_init_ims))
    # renderer.composite(osp.join(eval_dir, f'render_gt.png'), np.array(all_gt), np.array(all_cond_text), np.array(all_inits), np.array(all_init_ims))

[5 5 5]
torch.Size([3])
[[0]
 [0]
 [0]] (3, 1)
(3, 32, 8, 5, 26)

x.shape torch.Size([3, 32, 1041]) torch.float32 cond.shape torch.Size([3]) torch.int64 dummy_cond.shape torch.Size([3]) torch.int64 cond_obs.shape torch.Size([3, 32, 8, 5, 26]) torch.float32 time.shape torch.Size([3])
torch.Size([3, 128]) torch.Size([3, 8]) torch.Size([3, 1040])
x.shape torch.Size([3, 32, 1041]) torch.float32 cond.shape torch.Size([3]) torch.int64 dummy_cond.shape torch.Size([3]) torch.int64 cond_obs.shape torch.Size([3, 32, 8, 5, 26]) torch.float32 time.shape torch.Size([3])
torch.Size([3, 128]) torch.Size([3, 8]) torch.Size([3, 1040])
                                                                                                    
1 / 100 [                                                            ]   1% | 12.8 Hz
t : 99 | vmax : 0.0 | vmin : 0.0
x.shape torch.Size([3, 32, 1041]) torch.float32 cond.shape torch.Size([3]) torch.int64 dummy_cond.shape torch.Size([3]) torch.int64 cond_obs.shape torch.S

EOFError: 

In [ ]:
device = th.device('cpu' if not th.cuda.is_available() else 'cuda')

# learn_concept(envs, basedir, diffusion, dataset, renderer, "bc_proxy", 1.0, device)
open_loop_overcooked(envs, basedir, diffusion, dataset, renderer, "mep1_final", 1.0, device)

In [ ]:
dataset